In [1]:
import os
import requests
import pandas as pd

from langchain_xinference.chat_models import ChatXinference
from langchain_xinference.embeddings import XinferenceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import EvaluationDataset, SingleTurnSample
from ragas import evaluate, metrics
from tqdm import tqdm
from loguru import logger
from dotenv import load_dotenv

# 读取 .env 环境变量
load_dotenv()

True

In [2]:
# 读取 JSONL 文件
logger.info('Loading JSONL file...')
df = pd.read_json("qac_test.jsonl", lines=True, encoding="utf-8")
logger.success("JSONL file is read.")

# 创建 List 存储 QAC 数据
logger.info('Creating QAC list...')
qac_list = []
for index, row in df.iterrows():
    qac_list.append({
        "question": row["question"],
        "answer": row["answer"],
        "context": row["context"]
    })
logger.success("QAC list is created.")

2025-06-15 20:18:02.022 | INFO     | __main__:<module>:2 - Loading JSONL file...
2025-06-15 20:18:02.028 | SUCCESS  | __main__:<module>:4 - JSONL file is read.
2025-06-15 20:18:02.029 | INFO     | __main__:<module>:7 - Creating QAC list...
2025-06-15 20:18:02.029 | SUCCESS  | __main__:<module>:15 - QAC list is created.


In [3]:
try:
    logger.info("Creating the LLM and Embeddings.")
    # 创建 LLM 和 Embeddings 模型
    evaluator_llm = LangchainLLMWrapper(
        ChatXinference(
            server_url=os.getenv(""),
            model_uid=os.getenv("")
        )
    )
    logger.success("LLM successfully created.")
    # 用于评估的嵌入模型
    evaluator_embeddings = LangchainEmbeddingsWrapper(
        XinferenceEmbeddings(
            server_url=os.getenv(""),
            model_uid=os.getenv("")
        )
    )
    logger.success("Embeddings successfully created.")
except Exception as e:
    logger.exception(f"Failed to initialize LLM or Embeddings: {e}")
    raise

2025-06-15 20:18:05.878 | INFO     | __main__:<module>:2 - Creating the LLM and Embeddings.
2025-06-15 20:18:06.241 | SUCCESS  | __main__:<module>:11 - LLM successfully created.
2025-06-15 20:18:06.257 | SUCCESS  | __main__:<module>:20 - Embeddings successfully created.


In [4]:
# 构建 Dify API 请求头
logger.info('Building Dify API request header...')
qac_url = os.getenv("DIFY_API_BASE")
qac_headers = {
    'content-type': 'application/json; charset=UTF-8',
    'Authorization': f'Bearer {os.getenv("DIFY_API_KEY")}',
}
logger.success("Dify API request header is built.")

# 提交 QAC 数据集
logger.info('Submitting QAC items to Dify...')
samples = []
for qac in tqdm(
    qac_list, desc="Submitting QAC items to Dify",
    unit="item", total=len(qac_list)
):
    # 构建 QAC API 请求参数
    qac_payload = {
        "inputs": {},
        "query": qac.get("question"),
        "response_mode": "blocking",
        "conversation_id": "",
        "user": "abc-123",
        "files": []
    }

    # 提取 metadata 里面 retriever_resources 中的 content 作为召回文本块
    response = requests.post(url=qac_url, headers=qac_headers, json=qac_payload, timeout=600)
    res = response.json()

    retriever_resources = res.get('metadata', {}).get('retriever_resources', [])
    retriever_context = [str(doc.get("content", "")).strip("\n") for doc in retriever_resources]
    retriever_answer = res.get('answer', "")

    # 创建 SingleTurnSample 对象
    sample = SingleTurnSample(
        user_input=str(qac.get("question")).strip("\n"),  # 用户输入的问题
        retrieved_contexts=retriever_context,  # AI 召回的相关文本
        response=str(retriever_answer).strip("\n"),  # AI 生成的回答
        reference_contexts=[str(qac.get("context")).strip("\n")],  # 人给出的正确召回片段
        reference=str(qac.get("answer")).strip("\n")  # 人给出的正确回答
    )

    # 将样本添加到列表
    samples.append(sample)
logger.success('Submitted QAC items to Dify.')

2025-06-15 20:18:10.280 | INFO     | __main__:<module>:2 - Building Dify API request header...
2025-06-15 20:18:10.281 | SUCCESS  | __main__:<module>:8 - Dify API request header is built.
2025-06-15 20:18:10.281 | INFO     | __main__:<module>:11 - Submitting QAC items to Dify...
Submitting QAC items to Dify: 100%|██████████| 1/1 [00:18<00:00, 18.25s/item]
2025-06-15 20:18:28.538 | SUCCESS  | __main__:<module>:46 - Submitted QAC items to Dify.


In [6]:
# 将最后的结果添加到 EvaluationDataset 中
eval_dataset = EvaluationDataset(samples=samples)
# 评估指标列表
metrics_list = [
    # 答案相关性，越是不完整或包含冗余信息的答案，得分越低
    metrics.answer_relevancy,
    # 忠实度/可信度，衡量了生成的答案与给定上下文的事实一致性
    metrics.faithfulness,
    # 上下文精度，评估所有在上下文中呈现的与基本事实相关的条目是否排名较高。
    metrics.context_precision,
    # 上下文召回率，衡量检索到的上下文与人类提供的真实答案的一致程度。
    metrics.context_recall,
]

# 评估数据集
logger.info('Evaluating QAC items...')
results = evaluate(
    dataset=eval_dataset,
    metrics=metrics_list,
    llm=evaluator_llm,
    embeddings=evaluator_embeddings
)
logger.success('Evaluated QAC items.')

# 保存结果到 CSV 文件
logger.info('Saving results to CSV file...')
results.to_pandas().to_csv("qac_results.csv")
logger.success('Results saved to CSV file.')

2025-06-15 20:50:11.725 | INFO     | __main__:<module>:13 - Evaluating QAC items...


Evaluating:   0%|          | 0/5 [00:00<?, ?it/s]

Exception raised in Job[1]: APIConnectionError(Connection error.)
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[0]: TimeoutError()
2025-06-15 20:53:12.930 | SUCCESS  | __main__:<module>:20 - Evaluated QAC items.
2025-06-15 20:53:12.931 | INFO     | __main__:<module>:23 - Saving results to CSV file...
2025-06-15 20:53:12.935 | SUCCESS  | __main__:<module>:25 - Results saved to CSV file.
